In [2]:
import pandas as pd
%matplotlib inline
import matplotlib.pyplot as pl #importa o pacote de plotagem
import matplotlib as mpl
mpl.rcParams['figure.dpi'] = 400
import seaborn as sns
import numpy as np
import plotly.express as px
import scipy.stats as stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy import stats
from scipy.stats import f_oneway
from statsmodels.formula.api import ols
from statsmodels.stats.anova import anova_lm
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from scipy.stats import friedmanchisquare, wilcoxon
from itertools import combinations
from statsmodels.stats.multitest import multipletests

c:\Users\thami\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (


In [3]:
df_dados = pd.read_parquet('C:/Users/thami/OneDrive/Desktop/Projeto PPDI/Dados/tratados/bovino_bubalinos_condenacao_ano.parquet')
print(df_dados.head(10))

     MES_ANO UF_PROCEDENCIA ESPECIE PARTE_AFETADA  \
0 2022-01-01             RJ  Bovino     Intestino   
1 2023-01-01             MT  Bovino        Fígado   
2 2023-01-01             PR  Bovino  Meia Carcaça   
3 2024-01-01             TO  Bovino       Esôfago   
4 2025-01-01             GO  Bovino      Estômago   
5 2026-01-01             MT  Bovino        Fígado   
6 2022-02-01             RO  Bovino        Pulmão   
7 2023-02-01             MS  Bovino  Meia Carcaça   
8 2024-02-01             RS  Bovino          rabo   
9 2025-02-01             GO  Bovino        Pulmão   

                                         DIAGNOSTICO  \
0             CONTAMINAÇÃO GASTROINTESTINAL E BILIAR   
1             ESTADO ANORMAL/PATOLÓGICO NÃO PREVISTO   
2                               ABSCESSO (MAMÍFEROS)   
3                   ALTERAÇÃO LINFÁTICA INESPECÍFICA   
4  NEOPLASIA                                     ...   
5                   ALTERAÇÃO LINFÁTICA INESPECÍFICA   
6          EQUINOCOCOSE/

In [4]:
df_dados_Cond = pd.read_parquet('C:/Users/thami/OneDrive/Desktop/Projeto PPDI/Dados/tratados/bovino_bubalinos_abate_país.parquet')
print(df_dados_Cond.head(10))

    ANO  MES UF_PROCEDENCIA           MUNICIPIO_PROCEDENCIA CATEGORIA ESPECIE  \
0  2022    1             AC                       Brasileia    Bovino  Bovino   
1  2022    1             AC                  Epitaciolandia    Bovino  Bovino   
2  2021    2             GO                      Mossamedes    Bovino  Bovino   
3  2021    4             GO                           Araçu    Bovino  Bovino   
4  2021    4             GO                          Itauçu    Bovino  Bovino   
5  2021    4             GO                        Itaberai    Bovino  Bovino   
6  2021    5             GO  Sem correspondência na PGA GTA    Bovino  Bovino   
7  2021    7             GO  Sem correspondência na PGA GTA    Bovino  Bovino   
8  2021    8             GO                         Ipameri    Bovino  Bovino   
9  2021   11             GO                       Matrincha    Bovino  Bovino   

            Sexo  QUANTIDADE  
0  Indeterminado        3335  
1  Indeterminado        1962  
2  Indeterminad

In [14]:
df_dados_partes_2 = df_dados[df_dados['PARTE_AFETADA'].isin(['Carcaça','Meia Carcaça'])]
df_dados_partes_2.groupby(['PARTE_AFETADA', 'ANO'])['TOTAL_PARTES_AFETADAS'].sum()

PARTE_AFETADA  ANO 
Carcaça        2021     83551
               2022     69486
               2023    144779
               2024    174852
               2025    189613
               2026    182487
Meia Carcaça   2021      5003
               2022      9922
               2023      9572
               2024     14979
               2025     17056
               2026      4299
Name: TOTAL_PARTES_AFETADAS, dtype: int64

In [5]:
df_dados_diag = df_dados["DIAGNOSTICO"].unique()
sorted(df_dados_diag)

['ABORTO ENZOÓTICO DAS OVELHAS (CONFIRMADO)',
 'ABORTO RECENTE (FÊMEA)',
 'ABSCESSO',
 'ABSCESSO (MAMÍFEROS)',
 'ABSCESSO SUBCUTÂNEO (LAGOMORFOS)',
 'ACROBUSTITE                                       ',
 'ACTINOBACILOSE (RUMINANTES)',
 'ACTINOMICOSE (JÁ NOTIFICADO)',
 'ACTINOMICOSE (NOTIFICAÇÃO SIF)',
 'ADENITE                                           ',
 'ADERENCIAS                                        ',
 'ADERÊNCIA DE PLEURA (PURULENTA)',
 'ADERÊNCIA DE PLEURA (SECA)',
 'ADERÊNCIA DE PLEURA (ÚMIDA)',
 'ADIPOXANTOSE                                      ',
 'AFECÇÕES DO APARELHO REPRODUTOR (COELHO)',
 'AGALAXIA CONTAGIOSA (JÁ NOTIFICADO)',
 'AGALAXIA CONTAGIOSA (NOTIFICAÇÃO SIF)',
 'ALTERAÇÃO LINFÁTICA INESPECÍFICA',
 'ALTERAÇÃO RESTRITA',
 'ALTERAÇÕES MUSCULARES',
 'ALTERAÇÕES MUSCULARES (ESTRESSE/FADIGA)',
 'ALTERAÇÕES MUSCULARES (HEMORRAGIAS)',
 'ALTERAÇÕES MUSCULARES (RIGIDEZ ATÍPICA)',
 'ANEMIA                                            ',
 'ARTRITE                            

In [6]:
df_dados_doenca = df_dados[df_dados["DIAGNOSTICO"].isin(["BRUCELLA ABORTUS (JÁ NOTIFICADO)", "BRUCELLA ABORTUS (NOTIFICAÇÃO SIF)",
                                                         "BRUCELLA MELITENSIS (CONFIRMADO)", "BRUCELLA MELITENSIS (SUSPEITO)",
                                                         "BRUCELOSE", "BRUCELOSE (REAGENTES JÁ NOTIFICADOS)", "BRUCELOSE (REAGENTES NOTIFICAÇÃO SIF)",
                                                         "CISTICERCOSE", 'CISTICERCOSE BOVINA', 'CISTICERCOSE CALCIFICADA', 'CISTICERCOSE OVINA',
                                                         'CISTICERCOSE SUÍNA (NOTIFICAÇÃO SIF)', 'CISTICERCOSE VIVA',
                                                         'FASCIOLOSE', 'FASCIOLOSE HEPÁTICA (NOTIFICAÇÃO SIF)', 'LISTERIOSE (JÁ NOTIFICADO)', 'HIDATIDOSE',
                                                         'RAIVA BOVINA (CONFIRMADO)', 'FEBRE Q (NOTIFICAÇÃO SIF)', 'SARCOCISTOSE', 'ESTOMATITE', 'ESTOMATITE VESICULAR (CONFIRMADO)', 
                                                         'TUBERCULOSE', 'TUBERCULOSE (JÁ NOTIFICADO)', 'TUBERCULOSE (NOTIFICAÇÃO SIF)', 'TUBERCULOSE (REAGENTE JÁ NOTIFICADO)',
                                                         'TUBERCULOSE (REAGENTE NOTIFICAÇÃO SIF)', 'TUBERCULOSE AVIÁRIA (NOTIFICAÇÃO SIF)', 'TUBERCULOSE CALCIFICADA', 
                                                         'TUBERCULOSE CASEOSA','TUBERCULOSE MILIAR', 'ABSCESSO', 'ABSCESSO (MAMÍFEROS)','CONTUSÃO'])].copy()
print(df_dados_doenca.head(10))

      MES_ANO UF_PROCEDENCIA ESPECIE PARTE_AFETADA  \
2  2023-01-01             PR  Bovino  Meia Carcaça   
17 2024-03-01             MT  Bovino     Intestino   
27 2024-04-01             MG  Bovino       Carcaça   
34 2023-05-01             RS  Bovino        Cabeça   
37 2024-05-01             PA  Bovino          Rins   
39 2025-05-01             PE  Bovino      Estômago   
40 2026-05-01             GO  Bovino          Rins   
43 2022-06-01             MG  Bovino       Carcaça   
49 2025-06-01             PR  Bovino        Pulmão   
53 2026-06-01             MG  Bovino       Coração   

                              DIAGNOSTICO  \
2                    ABSCESSO (MAMÍFEROS)   
17                    CISTICERCOSE BOVINA   
27                   ABSCESSO (MAMÍFEROS)   
34                    CISTICERCOSE BOVINA   
37  BRUCELOSE (REAGENTES NOTIFICAÇÃO SIF)   
39                    CISTICERCOSE BOVINA   
40          TUBERCULOSE (NOTIFICAÇÃO SIF)   
43          TUBERCULOSE (NOTIFICAÇÃO SIF)   


In [7]:
df_dados_doenca['doenca_padronizada'] = (
    df_dados_doenca['DIAGNOSTICO']
    .str.replace(r'\s*\(.*?\)', '', regex=True)
    .str.strip()
    .replace({
        'BRUCELLA ABORTUS': 'BRUCELOSE',
        'BRUCELLA MELITENSIS': 'BRUCELOSE',
        'BRUCELOSE': 'BRUCELOSE',
        'CISTICERCOSE BOVINA': 'CISTICERCOSE',
        'CISTICERCOSE OVINA': 'CISTICERCOSE',
        'CISTICERCOSE SUÍNA': 'CISTICERCOSE',
        'CISTICERCOSE CALCIFICADA': 'CISTICERCOSE',
        'CISTICERCOSE VIVA': 'CISTICERCOSE',
        'FASCIOLOSE HEPÁTICA': 'FASCIOLOSE',
        'RAIVA BOVINA': 'RAIVA',
        'ESTOMATITE VESICULAR': 'ESTOMATITE',
        'TUBERCULOSE AVIÁRIA': 'TUBERCULOSE',
        'TUBERCULOSE CALCIFICADA': 'TUBERCULOSE',
        'TUBERCULOSE CASEOSA': 'TUBERCULOSE',
        'TUBERCULOSE MILIAR': 'TUBERCULOSE'
    })
)
print(df_dados_doenca.head(10))

      MES_ANO UF_PROCEDENCIA ESPECIE PARTE_AFETADA  \
2  2023-01-01             PR  Bovino  Meia Carcaça   
17 2024-03-01             MT  Bovino     Intestino   
27 2024-04-01             MG  Bovino       Carcaça   
34 2023-05-01             RS  Bovino        Cabeça   
37 2024-05-01             PA  Bovino          Rins   
39 2025-05-01             PE  Bovino      Estômago   
40 2026-05-01             GO  Bovino          Rins   
43 2022-06-01             MG  Bovino       Carcaça   
49 2025-06-01             PR  Bovino        Pulmão   
53 2026-06-01             MG  Bovino       Coração   

                              DIAGNOSTICO  \
2                    ABSCESSO (MAMÍFEROS)   
17                    CISTICERCOSE BOVINA   
27                   ABSCESSO (MAMÍFEROS)   
34                    CISTICERCOSE BOVINA   
37  BRUCELOSE (REAGENTES NOTIFICAÇÃO SIF)   
39                    CISTICERCOSE BOVINA   
40          TUBERCULOSE (NOTIFICAÇÃO SIF)   
43          TUBERCULOSE (NOTIFICAÇÃO SIF)   


<h3>Análise por grupo de condenação das partes afetadas

In [8]:
df_dados_partes = df_dados_doenca[df_dados_doenca['PARTE_AFETADA'].isin(['Carcaça','Meia Carcaça'])]

In [9]:
df_dados_partes.groupby(['PARTE_AFETADA', 'doenca_padronizada']).size().reset_index(name='count').sort_values(by='count', ascending=False)

,PARTE_AFETADA,doenca_padronizada,count
0,Carcaça,ABSCESSO,4189
2,Carcaça,CISTICERCOSE,3006
7,Carcaça,TUBERCULOSE,1488
8,Meia Carcaça,ABSCESSO,1036
3,Carcaça,FASCIOLOSE,260
1,Carcaça,BRUCELOSE,225
10,Meia Carcaça,CISTICERCOSE,85
11,Meia Carcaça,TUBERCULOSE,5
9,Meia Carcaça,BRUCELOSE,2
4,Carcaça,FEBRE Q,1


In [10]:
df_dados_doenca.groupby(['doenca_padronizada','PARTE_AFETADA'], as_index=False)['TOTAL_PARTES_AFETADAS'].sum()

,doenca_padronizada,PARTE_AFETADA,TOTAL_PARTES_AFETADAS
0,ABSCESSO,Baço,3325
1,ABSCESSO,Cabeça,5930
2,ABSCESSO,Carcaça,17286
3,ABSCESSO,Cauda(Rabo),2270
4,ABSCESSO,Coração,3710
...,...,...,...
127,TUBERCULOSE,Tendão,5
128,TUBERCULOSE,Vergalho,1
129,TUBERCULOSE,rabo,122
130,TUBERCULOSE,Úbere,298


In [16]:
cond_ano = (
    df_dados_doenca.groupby(['doenca_padronizada', 'ANO'], as_index=False).agg(CONDENADOS=('TOTAL_PARTES_AFETADAS', 'sum')))

abate_ano = (
    df_dados_Cond.groupby(['ANO'], as_index=False).agg(ABATE=('QUANTIDADE', 'sum')))

df_prev = cond_ano.merge(abate_ano, on='ANO', how='inner')
df_prev['PREVALENCIA'] = (df_prev['CONDENADOS'] / df_prev['ABATE']) * 100

tabela_prevalencia = df_prev.pivot(index='ANO', columns='doenca_padronizada', values='PREVALENCIA').fillna(0)

total_cond_parte = df_prev.groupby('doenca_padronizada')['CONDENADOS'].sum()
total_abate_geral = abate_ano['ABATE'].sum()

tabela_prevalencia['Total'] = total_cond_parte / total_abate_geral * 100

# Formata os números com 4 casas decimais e vírgula
tabela_formatada = tabela_prevalencia.apply(
    lambda col: col.map(lambda x: f"{x:.4f}".replace(".", ","))
)

tabela_formatada

doenca_padronizada,ABSCESSO,BRUCELOSE,CISTICERCOSE,ESTOMATITE,FASCIOLOSE,FEBRE Q,LISTERIOSE,RAIVA,TUBERCULOSE,Total
ANO,,,,,,,,,,
2021,"0,1591","0,0069","0,1341","0,0000","0,1673","0,0000","0,0000","0,0000","0,0102",nan
2022,"0,1609","0,0023","0,1520","0,0000","0,1410","0,0000","0,0000","0,0000","0,0107",nan
2023,"0,1472","0,0050","0,1417","0,0000","0,1450","0,0000","0,0000","0,0000","0,0053",nan
2024,"0,0561","0,0009","0,0826","0,0000","0,1332","0,0000","0,0000","0,0000","0,0062",nan
2025,"0,0563","0,0009","0,0667","0,0000","0,1436","0,0000","0,0000","0,0000","0,0076",nan
2026,"0,0484","0,0013","0,0569","0,0000","0,1085","0,0000","0,0000","0,0000","0,0203",nan


<h3>Análise brucelose

In [7]:
df_dados_brucelose = df_dados_doenca[df_dados_doenca["doenca_padronizada"].isin(["BRUCELOSE"])]
print(df_dados_brucelose.head(10))

        MES_ANO UF_PROCEDENCIA ESPECIE PARTE_AFETADA  \
37   2024-05-01             PA  Bovino          Rins   
49   2025-06-01             PR  Bovino        Pulmão   
150  2021-07-01             PA  Bovino       Coração   
721  2023-01-01             BA  Bovino      Estômago   
895  2023-01-01             RR  Bovino     Intestino   
1335 2025-05-01             TO  Bovino        Língua   
1583 2024-08-01             SC  Bovino       Carcaça   
1591 2021-09-01             PA  Bovino     Intestino   
1785 2026-08-01             RS  Bovino       Esôfago   
2786 2024-03-01             TO  Bovino        Pulmão   

                                DIAGNOSTICO              DESTINO_CONDENACAO  \
37    BRUCELOSE (REAGENTES NOTIFICAÇÃO SIF)        CONDENAÇÃO (POST MORTEM)   
49           BRUCELLA MELITENSIS (SUSPEITO)                       LIBERAÇÃO   
150   BRUCELOSE (REAGENTES NOTIFICAÇÃO SIF)        CONDENAÇÃO (POST MORTEM)   
721   BRUCELOSE (REAGENTES NOTIFICAÇÃO SIF)  CONDENAÇÃO TOTAL (POST

In [20]:
cond_ano = (
    df_dados_brucelose.groupby(['PARTE_AFETADA', 'ANO'], as_index=False).agg(CONDENADOS=('TOTAL_PARTES_AFETADAS', 'sum')))

cond_ano['ANO'] = cond_ano['ANO'].astype(int)
cond_ano['PARTE_AFETADA'] = cond_ano['PARTE_AFETADA'].astype(str).str.strip()

abate_ano = (
    df_dados_Cond.groupby(['ANO'], as_index=False).agg(ABATE=('QUANTIDADE', 'sum')))

abate_ano['ANO'] = abate_ano['ANO'].astype(int)

df_prev = cond_ano.merge(abate_ano, on='ANO', how='inner')
df_prev['PREVALENCIA'] = (df_prev['CONDENADOS'] / df_prev['ABATE']) * 100

tabela_prevalencia = df_prev.pivot_table(index='PARTE_AFETADA', columns='ANO', values='PREVALENCIA', aggfunc='sum',fill_value=0)

tabela_prevalencia = tabela_prevalencia.reindex(sorted(tabela_prevalencia.columns), axis=1)

tabela_prevalencia["Total"] = (
    total_cond_parte.reindex(tabela_prevalencia.index, fill_value=0) / total_abate_geral
)

# Remove o nome do eixo das colunas ("ANO") para um visual limpo
tabela_prevalencia.columns.name = None
tabela_prevalencia.index.name = None

# 6. Formata com 4 casas decimais e vírgula (padrão brasileiro)
tabela_formatada = tabela_prevalencia.apply(
    lambda col: col.map(lambda x: f"{x:.4f}".replace(".", ","))
)

# Exibe no VS Code (Notebook ou Terminal)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)
tabela_formatada

,2021,2022,2023,2024,2025,2026,Total
Baço,"0,0002","0,0001","0,0001","0,0000","0,0001","0,0001","0,0000"
Cabeça,"0,0005","0,0000","0,0002","0,0001","0,0000","0,0001","0,0000"
Carcaça,"0,0013","0,0003","0,0004","0,0002","0,0002","0,0002","0,0000"
Cauda(Rabo),"0,0002","0,0000","0,0000","0,0000","0,0000","0,0001","0,0000"
Coração,"0,0005","0,0002","0,0005","0,0001","0,0001","0,0001","0,0000"
Diafragma,"0,0001","0,0000","0,0000","0,0000","0,0000","0,0001","0,0000"
Estômago,"0,0005","0,0002","0,0004","0,0001","0,0001","0,0001","0,0000"
Esôfago,"0,0002","0,0001","0,0004","0,0000","0,0000","0,0000","0,0000"
Fígado,"0,0005","0,0002","0,0005","0,0001","0,0001","0,0000","0,0000"
Intestino,"0,0005","0,0002","0,0005","0,0001","0,0001","0,0001","0,0000"


<h3>tuberculose

In [21]:
df_dados_tuberculose = df_dados_doenca[df_dados_doenca["doenca_padronizada"].isin(["TUBERCULOSE"])]
print(df_dados_tuberculose.head(10))

       MES_ANO UF_PROCEDENCIA ESPECIE PARTE_AFETADA                    DIAGNOSTICO                                 DESTINO_CONDENACAO  TOTAL_PARTES_AFETADAS   ANO  MES doenca_padronizada
40  2026-05-01             GO  Bovino          Rins  TUBERCULOSE (NOTIFICAÇÃO SIF)  ESTERILIZAÇÃO POR CALOR ÚMIDO (SEGUIDA DE RESF...                      2  2026    5        TUBERCULOSE
43  2022-06-01             MG  Bovino       Carcaça  TUBERCULOSE (NOTIFICAÇÃO SIF)                                          LIBERAÇÃO                      1  2022    6        TUBERCULOSE
54  2021-07-01             SC  Bovino       Carcaça    TUBERCULOSE (JÁ NOTIFICADO)  ESTERILIZAÇÃO POR CALOR ÚMIDO (SEGUIDA DE RESF...                      2  2021    7        TUBERCULOSE
81  2021-10-01             MT  Bovino       Carcaça  TUBERCULOSE (NOTIFICAÇÃO SIF)                     CONDENAÇÃO TOTAL (POST MORTEM)                      3  2021   10        TUBERCULOSE
114 2025-03-01             RO  Bovino       Carcaça  TUBERCULOSE 

In [22]:
cond_ano = (
    df_dados_tuberculose.groupby(['PARTE_AFETADA', 'ANO'], as_index=False).agg(CONDENADOS=('TOTAL_PARTES_AFETADAS', 'sum')))

cond_ano['ANO'] = cond_ano['ANO'].astype(int)
cond_ano['PARTE_AFETADA'] = cond_ano['PARTE_AFETADA'].astype(str).str.strip()

abate_ano = (
    df_dados_Cond.groupby(['ANO'], as_index=False).agg(ABATE=('QUANTIDADE', 'sum')))

abate_ano['ANO'] = abate_ano['ANO'].astype(int)

df_prev = cond_ano.merge(abate_ano, on='ANO', how='inner')
df_prev['PREVALENCIA'] = (df_prev['CONDENADOS'] / df_prev['ABATE']) * 100

tabela_prevalencia = df_prev.pivot_table(index='PARTE_AFETADA', columns='ANO', values='PREVALENCIA', aggfunc='sum',fill_value=0)

tabela_prevalencia = tabela_prevalencia.reindex(sorted(tabela_prevalencia.columns), axis=1)

tabela_prevalencia["Total"] = (
    total_cond_parte.reindex(tabela_prevalencia.index, fill_value=0) / total_abate_geral
)

# Remove o nome do eixo das colunas ("ANO") para um visual limpo
tabela_prevalencia.columns.name = None
tabela_prevalencia.index.name = None

# 6. Formata com 4 casas decimais e vírgula (padrão brasileiro)
tabela_formatada = tabela_prevalencia.apply(
    lambda col: col.map(lambda x: f"{x:.4f}".replace(".", ","))
)

# Exibe no VS Code (Notebook ou Terminal)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)
tabela_formatada

,2021,2022,2023,2024,2025,2026,Total
Baço,"0,0004","0,0005","0,0002","0,0002","0,0004","0,0011","0,0000"
Cabeça,"0,0005","0,0008","0,0003","0,0004","0,0005","0,0013","0,0000"
Carcaça,"0,0038","0,0027","0,0020","0,0020","0,0013","0,0021","0,0000"
Cauda(Rabo),"0,0004","0,0003","0,0002","0,0002","0,0004","0,0006","0,0000"
Coração,"0,0004","0,0006","0,0002","0,0002","0,0004","0,0012","0,0000"
Cérebro(Miolo),"0,0000","0,0000","0,0000","0,0000","0,0000","0,0000","0,0000"
Diafragma,"0,0002","0,0006","0,0001","0,0003","0,0004","0,0012","0,0000"
Estômago,"0,0004","0,0004","0,0002","0,0003","0,0004","0,0011","0,0000"
Esôfago,"0,0002","0,0002","0,0001","0,0001","0,0004","0,0011","0,0000"
Fígado,"0,0005","0,0007","0,0003","0,0004","0,0005","0,0013","0,0000"


<h3>Cisticercose

In [23]:
df_dados_cisticercose = df_dados_doenca[df_dados_doenca["doenca_padronizada"].isin(["CISTICERCOSE"])]
print(df_dados_cisticercose.head(10))

       MES_ANO UF_PROCEDENCIA ESPECIE PARTE_AFETADA          DIAGNOSTICO                                 DESTINO_CONDENACAO  TOTAL_PARTES_AFETADAS   ANO  MES doenca_padronizada
17  2024-03-01             MT  Bovino     Intestino  CISTICERCOSE BOVINA                           CONDENAÇÃO (POST MORTEM)                      3  2024    3       CISTICERCOSE
34  2023-05-01             RS  Bovino        Cabeça  CISTICERCOSE BOVINA               TRATAMENTO PELO FRIO (-10ºC/10 DIAS)                      6  2023    5       CISTICERCOSE
39  2025-05-01             PE  Bovino      Estômago  CISTICERCOSE BOVINA       REMOÇÃO DA PARTE AFETADA (LINHA DE INSPEÇÃO)                      3  2025    5       CISTICERCOSE
53  2026-06-01             MG  Bovino       Coração  CISTICERCOSE BOVINA               TRATAMENTO PELO FRIO (-10ºC/10 DIAS)                     15  2026    6       CISTICERCOSE
60  2025-07-01             BA  Bovino        Pulmão  CISTICERCOSE BOVINA                           CONDENAÇÃO (POST

In [24]:
cond_ano = (
    df_dados_cisticercose.groupby(['PARTE_AFETADA', 'ANO'], as_index=False).agg(CONDENADOS=('TOTAL_PARTES_AFETADAS', 'sum')))

cond_ano['ANO'] = cond_ano['ANO'].astype(int)
cond_ano['PARTE_AFETADA'] = cond_ano['PARTE_AFETADA'].astype(str).str.strip()

abate_ano = (
    df_dados_Cond.groupby(['ANO'], as_index=False).agg(ABATE=('QUANTIDADE', 'sum')))

abate_ano['ANO'] = abate_ano['ANO'].astype(int)

df_prev = cond_ano.merge(abate_ano, on='ANO', how='inner')
df_prev['PREVALENCIA'] = (df_prev['CONDENADOS'] / df_prev['ABATE']) * 100

tabela_prevalencia = df_prev.pivot_table(index='PARTE_AFETADA', columns='ANO', values='PREVALENCIA', aggfunc='sum',fill_value=0)

tabela_prevalencia = tabela_prevalencia.reindex(sorted(tabela_prevalencia.columns), axis=1)

tabela_prevalencia["Total"] = (
    total_cond_parte.reindex(tabela_prevalencia.index, fill_value=0) / total_abate_geral
)

# Remove o nome do eixo das colunas ("ANO") para um visual limpo
tabela_prevalencia.columns.name = None
tabela_prevalencia.index.name = None

# 6. Formata com 4 casas decimais e vírgula (padrão brasileiro)
tabela_formatada = tabela_prevalencia.apply(
    lambda col: col.map(lambda x: f"{x:.4f}".replace(".", ","))
)

# Exibe no VS Code (Notebook ou Terminal)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)
tabela_formatada

,2021,2022,2023,2024,2025,2026,Total
Baço,"0,0012","0,0012","0,0018","0,0019","0,0032","0,0026","0,0000"
Cabeça,"0,0151","0,0155","0,0152","0,0083","0,0069","0,0065","0,0000"
Carcaça,"0,0300","0,0250","0,0157","0,0109","0,0077","0,0073","0,0000"
Cauda(Rabo),"0,0080","0,0093","0,0066","0,0032","0,0036","0,0030","0,0000"
Coração,"0,0219","0,0207","0,0201","0,0130","0,0097","0,0086","0,0000"
Cérebro(Miolo),"0,0002","0,0010","0,0007","0,0013","0,0008","0,0001","0,0000"
Diafragma,"0,0083","0,0089","0,0104","0,0057","0,0042","0,0036","0,0000"
Estômago,"0,0028","0,0053","0,0043","0,0023","0,0019","0,0011","0,0000"
Esôfago,"0,0075","0,0078","0,0052","0,0042","0,0037","0,0028","0,0000"
Fígado,"0,0116","0,0144","0,0136","0,0072","0,0067","0,0057","0,0000"
